# Characterisation of curated systems

In [6]:
import json
import os
import pandas as pd
import numpy as np
import shutil
import matplotlib.pyplot as plt
import seaborn as sns

while os.path.split(os.getcwd())[-1] != 'calvados-slim':
    os.chdir('..')

from src import curation, utils, visualisation

In [7]:
# --- Inputs ---
with open('data/curation/1dp-idp/systems.json') as file:
    systems = json.load(file)
EXCLUDE      = ['grasp1-grm5', 'u2af2-sf1', 'pp1c-ppp1r2']

# --- Configuration ---
SYSTEMS_DIR = os.path.abspath('data/curation/1dp-idp/systems')


In [104]:
characteristics = pd.DataFrame(columns=[
    'Acceptor size', 'Donor size',
    'Contacts',
    'Acceptor contact residues', 'Donor contact residues',
    'SLiM span',
])
characteristics.index.name = 'System'

for system in systems:

    if system['ID'] in EXCLUDE:
        continue

    slim = system['SLiMs'][0]
    for chain in system['Chains']:
        if chain['ID'] == slim['Acceptor ID']:
            acceptor_len = len(chain['Sequence'])
        if chain['ID'] == slim['Donor ID']:
            donor_len = len(chain['Sequence'])

    contacts = utils.parse_specific_interactions(os.path.join(SYSTEMS_DIR, system['ID'], 'specific_interactions.txt'))
    n_c = len(contacts)

    n_acr = contacts['Acceptor residue'].nunique()
    n_dcr = contacts['Donor residue'].nunique()

    n_dcs = contacts['Donor residue'].max() - contacts['Donor residue'].min() + 1

    characteristics.loc[system['ID']] = [acceptor_len, donor_len, n_c, n_acr, n_dcr, n_dcs]

characteristics.to_csv('data/analysis/characteristics.csv')